In [108]:
import os
import json
# Load queries
path = os.getcwd()
with open(os.path.join(path, "data_for_git/factscore_bio.jsonl"), "r", encoding="utf-8") as f:
    queries = [json.loads(line) for line in f]

with open(os.path.join(path, "data_for_git/query_retrieval_top4.json"), "r", encoding="utf-8") as f:
    retrieval = json.loads(f.read())


queries = [query["prompt"] for query in queries]
# queries = queries[:5]  # Uncomment for quick testing
retrievals = [[doc for doc in retrieval[x]] for x in queries]

In [109]:
system_prompt = """You are an AI assistant that provides biographical information based ONLY on the provided Wikipedia context.

IMPORTANT RULES:
- Base your answer EXCLUSIVELY on the retrieved documents
- If the retrieved documents do not contain information about the requested person, then you should clearly state that the retrieved documents do not contain information about the requested person.
- Do NOT use your general knowledge or make up information
- Only mention facts that are explicitly stated in the provided context
- You may provide detailed answers if relevant information is available in the retrieved documents
- The retrieved documents are held within the <CONTEXT> tags.
"""


# Prepare prompts as chat messages for vLLM API
full_prompts = []
m = 0
n = None
for query, retrievals in zip(queries[m:n], retrievals[m:n]):
    user_prompt = f"Retrieved context:\n <CONTEXT>"
    for i, retrieval in enumerate(retrievals):
        title = retrieval["title"]
        content = retrieval["contents"]
        user_prompt += f"START OF DOCUMENT {i} WITH TITLE: {title}\n{content}\n"
        user_prompt += f"\n ================END OF DOCUMENT {i} ================"
    user_prompt += f"<CONTEXT>\nBased on the documents above, i now want you to: {query}, ANSWER:"
    full_prompts.append([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ])

In [110]:
# Generation parameters
n_responses = 5

# Clear the output file (start fresh)
output_file = os.path.join(path, "data_for_git/responses.jsonl")
with open(output_file, "w") as f:
    pass  # Just create/clear the file


In [111]:
from openai.types.chat import ChatCompletion

def gnmt_length_penalty(length: int, alpha: float = 0.6) -> float:
    """
    Calculates the GNMT length penalty term.
    
    Args:
        length (int): The number of items in the sequence (tokens or facts).
        alpha (float): The penalty strength. 
                       1.0 = Standard average (1/L).
                       0.6 = Standard GNMT "sweet spot".
                       0.0 = No penalty (sum only).
    
    Returns:
        float: The penalty factor to DIVIDE the score by.
    """
    return ((5 + length) ** alpha) / ((5 + 1) ** alpha)




def get_sequence_logprobs(output: ChatCompletion):
    """
    Get the log probabilities of a sequence of tokens from a logprobs array.
    """
    logprobs_list = output.choices[0].logprobs
    sequence_logprob = 0
    token_count = 0
    for token_data in logprobs_list.content:
        token_lp = token_data.logprob
        
        # Sum it up manually
        sequence_logprob += token_lp
        token_count += 1
    return sequence_logprob / token_count / gnmt_length_penalty(token_count)



In [ ]:
from openai import OpenAI
import threading
import json

# Initialize vLLM client (OpenAI-compatible API)
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy-key"  # vLLM doesn't require auth, but OpenAI client needs a key
)
max_concurrent_requests = 400  # Limit how many requests are in flight at once
print(f"Generating {n_responses} responses for {len(full_prompts)} prompts...")
print(f"Max concurrent requests: {max_concurrent_requests}")

# Semaphore to limit concurrent requests
request_semaphore = threading.Semaphore(max_concurrent_requests)

# Thread-safe storage for responses
responses_lock = threading.Lock()
responses_dict = {}

def generate_response(prompt_messages, prompt_key, response_idx):
    """Generate a single response in a thread - vLLM handles batching automatically"""
    # Acquire semaphore - blocks if too many requests are active
    request_semaphore.acquire()
    try:
        completion = client.chat.completions.create(
            model="Qwen/Qwen2.5-7B-Instruct",  # Model name doesn't matter for vLLM
            messages=prompt_messages,
            temperature=0.7,
            top_p=0.9,
            max_tokens=1024,
            logprobs=True
        )
        response_text = completion.choices[0].message.content
        
        # Thread-safe append to responses
        with responses_lock:
            if prompt_key not in responses_dict:
                responses_dict[prompt_key] = []
            responses_dict[prompt_key].append({"response": response_text, "logprobs": get_sequence_logprobs(completion)})
            
            # Write to file when we have all responses for this prompt
            if len(responses_dict[prompt_key]) == n_responses:
                item = {"prompt": prompt_key, "responses": responses_dict[prompt_key]}
                with open(output_file, "a") as f:
                    f.write(json.dumps(item) + "\n")
    except Exception as e:
        print(f"Error generating response {response_idx} for prompt: {e}")
    finally:
        # Always release semaphore when done (success or failure)
        request_semaphore.release()

# Create all threads at once - semaphore will limit how many run concurrently
all_threads = []
for prompt_messages in full_prompts:
    prompt_key = prompt_messages[1]["content"]  # user message content
    
    # Create threads for n_responses requests per prompt
    for i in range(n_responses):
        thread = threading.Thread(
            target=generate_response,
            args=(prompt_messages, prompt_key, i)
        )
        thread.start()
        all_threads.append(thread)

# Wait for all requests to complete
for thread in all_threads:
    thread.join()

print(f"Saved responses to {output_file}")
print(f"Total prompts processed: {len(full_prompts)}")


Generating 5 responses for 683 prompts...
Max concurrent requests: 400
